In [0]:
from pyspark.sql.functions import trim, col, StringType
import pyspark.sql.functions as F
from pyspark.sql import functions as F

df = spark.table("dev_project.bronze.crm_sales_details")

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

df = (
    df.withColumn(
        "sls_order_dt",
        F.when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8), None
        ).otherwise(F.to_date(col("sls_order_dt").cast("string"), "yyyyMMdd")),
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8), None
        ).otherwise(F.to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd")),
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8), None
        ).otherwise(F.to_date(col("sls_due_dt").cast("string"), "yyyyMMdd")),
    )
)

df = df.withColumn(
    "sls_price",
    F.when(
        (col("sls_price").isNull()) | (col("sls_price") <= 0),
        F.when(
            col("sls_quantity") != 0, col("sls_sales") / col("sls_quantity")
        ).otherwise(None),
    ).otherwise(col("sls_price")),
)

name_map = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_key",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales",
    "sls_quantity": "quantity",
    "sls_price": "price",
}

for old_name, new_name in name_map.items():
    df = df.withColumnRenamed(old_name, new_name)
df.display()

df.write.mode("overwrite").saveAsTable("dev_project.silver.crm_sales")